# 📱 Build CENAD APK sur Google Colab
**Instructions :**
1. Exécutez chaque cellule dans l'ordre (Ctrl+Enter)
2. À l'étape 3, uploadez votre ZIP du projet
3. La compilation dure ~20-30 minutes (première fois)
4. Téléchargez l'APK à la fin

> ⚠️ Connectez-vous avec un compte Google. Runtime > Changer le type > T4 GPU (ou CPU suffit)

In [ ]:
# ── ÉTAPE 1 : Dépendances système ──
%%bash
apt-get update -qq
apt-get install -y -qq \
  git zip unzip \
  openjdk-17-jdk \
  autoconf libtool pkg-config \
  zlib1g-dev libncurses5-dev libncursesw5-dev \
  cmake libffi-dev libssl-dev \
  build-essential ccache
echo '✅ Dépendances système installées'

In [ ]:
# ── ÉTAPE 2 : Installer Buildozer ──
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'buildozer', 'cython==0.29.33'])
print('✅ Buildozer installé')
!buildozer version

In [ ]:
# ── ÉTAPE 3 : Uploader le projet ──
# Uploadez votre fichier cenad_app.zip
from google.colab import files
import zipfile, os

print('Sélectionnez votre ZIP du projet CENAD...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs('/content/cenad_app', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/cenad_app')

# Lister les fichiers extraits
!ls /content/cenad_app/
print('✅ Projet extrait')

In [ ]:
# ── ÉTAPE 4 : Vérifier buildozer.spec ──
import os

spec_path = '/content/cenad_app/buildozer.spec'
if not os.path.exists(spec_path):
    # Créer un spec minimal si absent
    spec = """[app]
title = CENAD
package.name = cenad
package.domain = org.cenad
source.dir = .
source.include_exts = py,png,jpg,kv,atlas,db,csv
version = 1.0.0
requirements = python3,kivy==2.3.0,sqlite3,pillow,android
orientation = portrait
android.permissions = READ_EXTERNAL_STORAGE,WRITE_EXTERNAL_STORAGE,CALL_PHONE,SEND_SMS
android.minapi = 21
android.api = 33
android.ndk = 25b
android.archs = arm64-v8a, armeabi-v7a
android.accept_sdk_license = True
[buildozer]
log_level = 2
"""
    with open(spec_path, 'w') as f:
        f.write(spec)
    print('✅ buildozer.spec créé')
else:
    print('✅ buildozer.spec trouvé')
    !cat {spec_path}

In [ ]:
# ── ÉTAPE 5 : BUILD APK (20-30 min) ──
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

%cd /content/cenad_app
!buildozer -v android debug 2>&1 | tee /content/build.log
print('\n✅ Build terminé !')

In [ ]:
# ── ÉTAPE 6 : Télécharger l'APK ──
import glob
from google.colab import files

apks = glob.glob('/content/cenad_app/bin/*.apk')
if apks:
    apk = apks[0]
    size_mb = os.path.getsize(apk) / 1024 / 1024
    print(f'✅ APK trouvé : {apk} ({size_mb:.1f} MB)')
    files.download(apk)
else:
    print('❌ APK non trouvé. Vérifiez le log :')
    !tail -50 /content/build.log

In [ ]:
# ── BONUS : Voir les erreurs si le build a échoué ──
!grep -i 'error\|exception\|traceback' /content/build.log | tail -30